In [1]:
import os
import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================
CSV_PATH = "AnonDB.csv"

MAKE_COLUMN = "Make"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

RANDOM_SEED = 42


# =========================================================
# VALIDATE CONFIG
# =========================================================
ratios = np.array(
    [TRAIN_RATIO, VAL_RATIO, TEST_RATIO],
    dtype=float
)

if not np.isclose(ratios.sum(), 1.0):
    raise ValueError(
        f"TRAIN_RATIO + VAL_RATIO + TEST_RATIO must equal 1.0, "
        f"but currently equals {ratios.sum():.3f}"
    )

if np.any(ratios < 0):
    raise ValueError("Split ratios cannot be negative.")


# =========================================================
# OUTPUT FILE NAME
# =========================================================
train_pct = round(TRAIN_RATIO * 100)
val_pct = round(VAL_RATIO * 100)
test_pct = round(TEST_RATIO * 100)

csv_directory = os.path.dirname(os.path.abspath(CSV_PATH))
csv_filename = os.path.splitext(os.path.basename(CSV_PATH))[0]

OUTPUT_PATH = os.path.join(
    csv_directory,
    f"{csv_filename}_{train_pct}_{val_pct}_{test_pct}_seed{RANDOM_SEED}.csv"
)


# =========================================================
# DETERMINE COUNTS FOR EACH MAKE
# =========================================================
def calculate_split_counts(n_samples, split_ratios):
    """
    Calculate train/val/test counts for one Make.

    For a Make with at least 3 samples, this function ensures that
    at least one sample enters train, validation, and test.
    """
    raw_counts = n_samples * split_ratios
    counts = np.floor(raw_counts).astype(int)

    remaining = n_samples - counts.sum()

    # Give remaining samples to splits with largest fractional remainder
    fractional_parts = raw_counts - counts
    remainder_order = np.argsort(-fractional_parts)

    for i in remainder_order[:remaining]:
        counts[i] += 1

    active_splits = np.where(split_ratios > 0)[0]

    # Ensure every active split contains at least one sample,
    # provided that enough samples exist.
    if n_samples >= len(active_splits):
        for split_index in active_splits:
            if counts[split_index] == 0:
                donor_candidates = [
                    i for i in active_splits
                    if counts[i] > 1
                ]

                if donor_candidates:
                    donor_index = max(
                        donor_candidates,
                        key=lambda i: counts[i]
                    )

                    counts[donor_index] -= 1
                    counts[split_index] += 1

    return counts


# =========================================================
# ASSIGN SPLIT WITHIN EACH MAKE
# =========================================================
def assign_fixed_split(
    dataframe,
    make_column,
    train_ratio,
    val_ratio,
    test_ratio,
    seed
):
    """
    Shuffle and split samples independently within every Make.

    All original dataframe columns are retained.
    Only the new column 'split' is added.
    """
    result = dataframe.copy()

    # Remove an existing split column when rerunning the script
    if "split" in result.columns:
        result = result.drop(columns=["split"])

    result["split"] = pd.NA

    # Temporary grouping column; original Make column remains untouched
    make_for_split = (
        result[make_column]
        .astype("string")
        .fillna("Unknown")
    )

    split_ratios = np.array(
        [train_ratio, val_ratio, test_ratio],
        dtype=float
    )

    rng = np.random.default_rng(seed)

    make_values = sorted(make_for_split.unique().tolist())

    rare_makes = []

    for make_value in make_values:
        make_indices = result.index[
            make_for_split == make_value
        ].to_numpy()

        make_indices = rng.permutation(make_indices)

        n_samples = len(make_indices)

        n_train, n_val, n_test = calculate_split_counts(
            n_samples,
            split_ratios
        )

        if n_samples < 3:
            rare_makes.append((make_value, n_samples))

        train_end = n_train
        val_end = n_train + n_val

        train_indices = make_indices[:train_end]
        val_indices = make_indices[train_end:val_end]
        test_indices = make_indices[val_end:val_end + n_test]

        result.loc[train_indices, "split"] = "train"
        result.loc[val_indices, "split"] = "val"
        result.loc[test_indices, "split"] = "test"

    if result["split"].isna().any():
        missing_count = result["split"].isna().sum()

        raise RuntimeError(
            f"{missing_count} rows were not assigned to a split."
        )

    if rare_makes:
        print("\nWarning: some Make categories contain fewer than 3 samples.")
        print("They cannot appear in all three splits:")

        for make_value, count in rare_makes:
            print(f"- {make_value}: {count} sample(s)")

    return result


# =========================================================
# LOAD ORIGINAL CSV
# =========================================================
df = pd.read_csv(CSV_PATH)

if MAKE_COLUMN not in df.columns:
    raise ValueError(
        f"Column '{MAKE_COLUMN}' was not found in {CSV_PATH}."
    )

print(f"Original dataset rows: {len(df)}")
print(f"Number of Make categories: {df[MAKE_COLUMN].nunique(dropna=False)}")


# =========================================================
# CREATE FIXED SPLIT
# =========================================================
df_split = assign_fixed_split(
    dataframe=df,
    make_column=MAKE_COLUMN,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=RANDOM_SEED
)


# =========================================================
# CHECK RESULTS
# =========================================================
print("\nOverall split counts:")
print(
    df_split["split"]
    .value_counts()
    .reindex(["train", "val", "test"], fill_value=0)
)

print("\nOverall split proportions:")
print(
    df_split["split"]
    .value_counts(normalize=True)
    .reindex(["train", "val", "test"], fill_value=0)
)

print("\nSamples from each Make in each split:")

make_split_table = pd.crosstab(
    df_split[MAKE_COLUMN].fillna("Unknown"),
    df_split["split"]
).reindex(
    columns=["train", "val", "test"],
    fill_value=0
)

print(make_split_table)


# =========================================================
# SAVE ALL ORIGINAL COLUMNS + SPLIT COLUMN
# =========================================================
df_split.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nFixed split CSV saved to:")
print(OUTPUT_PATH)

print("\nThe final columns are:")
print(df_split.columns.tolist())

Original dataset rows: 616
Number of Make categories: 15

Overall split counts:
split
train    490
val       65
test      61
Name: count, dtype: int64

Overall split proportions:
split
train    0.795455
val      0.105519
test     0.099026
Name: proportion, dtype: float64

Samples from each Make in each split:
split  train  val  test
Make                   
5         62    8     8
7          7    1     1
12        68    9     8
15        39    5     5
17        38    5     5
20        40    5     5
22        41    5     5
24        38    5     5
25        54    7     7
30         8    1     1
32        21    3     2
34        20    3     2
35        21    3     3
36        13    2     2
38        20    3     2

Fixed split CSV saved to:
c:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\AnonDB_80_10_10_seed42.csv

The final columns are:
['Unnamed: 0', 'Mod_ID', 'Confidential', 'Make', 'Model', 'Interconnect_Tech', 'Module_Area_(cm2)', 'Junction_Box_Typ

In [ ]:
import os
import math
import copy
import random
import time
import gc

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models


# =========================================================
# CONFIG
# =========================================================
CSV_PATH = "AnonDB_80_10_10_seed42.csv"   # <-- split CSV
MAKE_COLUMN = "Make"

IMAGE_SIZE = 240
BATCH_SIZE = 16          # kalau OOM, ganti ke 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
BACKBONE_LR = 1e-5
RANDOM_SEED = 42
NUM_WORKERS = 0
PRELOAD_TO_RAM = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA used by PyTorch:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


# =========================================================
# REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)


# =========================================================
# HELPER FUNCTIONS
# =========================================================
def resolve_relative_path(path_str, base_dir, add_wrp=True):
    if pd.isna(path_str):
        return None

    path_str = str(path_str).strip()

    if path_str == "":
        return None

    path_str = path_str.replace("\\", os.sep).replace("/", os.sep)

    if add_wrp:
        root, ext = os.path.splitext(path_str)
        if not root.endswith("_wrp"):
            path_str = root + "_wrp" + ext

    if os.path.isabs(path_str):
        return os.path.normpath(path_str)

    if path_str.startswith("." + os.sep):
        path_str = path_str[2:]

    return os.path.normpath(os.path.join(base_dir, path_str))


def prepare_dataframe(csv_path):
    """
    Read split CSV, resolve image paths, compute pf_ratio, and keep
    the split column + Make column.
    """
    base_dir = os.path.dirname(os.path.abspath(csv_path))
    df = pd.read_csv(csv_path)

    required_cols = [
        "ELLPath",
        "ELHPath",
        "Pmp_(W)",
        "Nameplate_Pmp_(W)",
        "split",
        MAKE_COLUMN
    ]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in CSV: {col}")

    df["Pmp_(W)"] = pd.to_numeric(df["Pmp_(W)"], errors="coerce")
    df["Nameplate_Pmp_(W)"] = pd.to_numeric(df["Nameplate_Pmp_(W)"], errors="coerce")

    df = df[
        df["Nameplate_Pmp_(W)"].notna()
        & (df["Nameplate_Pmp_(W)"] > 0)
        & df["Pmp_(W)"].notna()
    ].copy()

    df["pf_ratio"] = df["Pmp_(W)"] / df["Nameplate_Pmp_(W)"]

    df["low_path"] = df["ELLPath"].apply(lambda p: resolve_relative_path(p, base_dir))
    df["high_path"] = df["ELHPath"].apply(lambda p: resolve_relative_path(p, base_dir))

    df = df[df["low_path"].notna() & df["high_path"].notna()].copy()

    df["low_exists"] = df["low_path"].apply(os.path.exists)
    df["high_exists"] = df["high_path"].apply(os.path.exists)

    total_before = len(df)
    df = df[df["low_exists"] & df["high_exists"]].copy()
    total_after = len(df)

    print(f"Rows with valid power data before file filtering: {total_before}")
    print(f"Rows with both EL files existing: {total_after}")
    print(f"Rows removed due to missing image file(s): {total_before - total_after}")

    df["split"] = df["split"].astype(str).str.strip().str.lower()
    df = df[df["split"].isin(["train", "val", "test"])].copy()

    df["make_plot"] = df[MAKE_COLUMN].fillna("Unknown").astype(str)

    optional_id_cols = [
        c for c in ["ID", "ModuleID", "Module", "Serial", "PanelID"]
        if c in df.columns
    ]

    keep_cols = optional_id_cols + [
        MAKE_COLUMN,
        "make_plot",
        "split",
        "low_path",
        "high_path",
        "pf_ratio"
    ]

    return df[keep_cols].reset_index(drop=True)


# =========================================================
# IMAGE PROCESSING
# =========================================================
def read_grayscale_tiff(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Failed to load image: {path}")

    return img


def resize_with_padding_gray(img, target_size=240):
    h, w = img.shape[:2]

    scale = target_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(
        img,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    canvas = np.zeros((target_size, target_size), dtype=np.uint8)

    y0 = (target_size - new_h) // 2
    x0 = (target_size - new_w) // 2

    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized

    return canvas


# =========================================================
# DATASET
# =========================================================
class DualELDataset(Dataset):
    def __init__(self, dataframe, image_size=240, augment=False, preload=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment
        self.cache = False

        if preload:
            self._preload_all()

    def _preload_all(self):
        n = len(self.df)
        expected_mb = n * 2 * self.image_size * self.image_size / (1024 ** 2)

        print(f"\nPreloading {n} samples ({n * 2} images) into RAM...")
        print(f"Expected memory: ~{expected_mb:.0f} MB")

        t0 = time.time()

        self.cache_low = np.empty(
            (n, self.image_size, self.image_size),
            dtype=np.uint8
        )

        self.cache_high = np.empty(
            (n, self.image_size, self.image_size),
            dtype=np.uint8
        )

        failed = 0

        for i in range(n):
            row = self.df.iloc[i]

            try:
                low_img = read_grayscale_tiff(row["low_path"])
                high_img = read_grayscale_tiff(row["high_path"])

                self.cache_low[i] = resize_with_padding_gray(low_img, self.image_size)
                self.cache_high[i] = resize_with_padding_gray(high_img, self.image_size)

            except Exception as e:
                print(f"Warning: failed to load sample {i}: {e}")
                self.cache_low[i] = 0
                self.cache_high[i] = 0
                failed += 1

        elapsed = time.time() - t0
        mem_mb = (self.cache_low.nbytes + self.cache_high.nbytes) / (1024 ** 2)

        print(
            f"Done in {elapsed:.1f}s | "
            f"RAM used: {mem_mb:.1f} MB | "
            f"Failed: {failed}"
        )

        self.cache = True

    def apply_same_augmentation(self, low_img, high_img):
        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 1)
            high_img = cv2.flip(high_img, 1)

        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 0)
            high_img = cv2.flip(high_img, 0)

        return low_img, high_img

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if self.cache:
            low_img = self.cache_low[idx].copy()
            high_img = self.cache_high[idx].copy()
        else:
            low_img = read_grayscale_tiff(row["low_path"])
            high_img = read_grayscale_tiff(row["high_path"])

            low_img = resize_with_padding_gray(low_img, self.image_size)
            high_img = resize_with_padding_gray(high_img, self.image_size)

        if self.augment:
            low_img, high_img = self.apply_same_augmentation(low_img, high_img)

        low_img = low_img.astype(np.float32) / 255.0
        high_img = high_img.astype(np.float32) / 255.0

        low_img = np.expand_dims(low_img, axis=0)
        high_img = np.expand_dims(high_img, axis=0)

        x = np.concatenate([low_img, high_img], axis=0)
        y = float(row["pf_ratio"])

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )


# =========================================================
# MODEL
# =========================================================
class BoundedOutput(nn.Module):
    def __init__(self, in_features, min_val=0.80, max_val=1.20):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)
        self.min_val = min_val
        self.max_val = max_val

    def forward(self, x):
        x = torch.sigmoid(self.linear(x))
        return x * (self.max_val - self.min_val) + self.min_val


def build_efficientnet_b1_2ch(pretrained=True):
    weights = models.EfficientNet_B1_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b1(weights=weights)

    old_conv = model.features[0][0]

    new_conv = nn.Conv2d(
        in_channels=2,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )

    with torch.no_grad():
        if pretrained:
            rgb_mean = old_conv.weight.mean(dim=1)
            new_conv.weight[:, 0, :, :] = rgb_mean
            new_conv.weight[:, 1, :, :] = rgb_mean
            new_conv.weight *= 0.5
        else:
            nn.init.kaiming_normal_(
                new_conv.weight,
                mode="fan_out",
                nonlinearity="relu"
            )

    model.features[0][0] = new_conv

    in_features = model.classifier[1].in_features
    model.classifier[1] = BoundedOutput(
        in_features,
        min_val=0.80,
        max_val=1.20
    )

    return model


# =========================================================
# TRAIN / EVALUATE
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    return running_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    preds = []
    trues = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device).unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)

            preds.extend(outputs.squeeze(1).cpu().numpy())
            trues.extend(targets.squeeze(1).cpu().numpy())

    preds = np.array(preds)
    trues = np.array(trues)

    loss_value = running_loss / len(loader.dataset)
    mae = mean_absolute_error(trues, preds)
    rmse = math.sqrt(mean_squared_error(trues, preds))
    r2 = r2_score(trues, preds)

    return loss_value, mae, rmse, r2, preds, trues


# =========================================================
# PLOTTING HELPERS
# =========================================================
def build_make_palette(make_values, cmap_name="tab20"):
    """
    Good categorical palette for many makes.
    """
    makes = sorted(pd.Series(make_values).fillna("Unknown").astype(str).unique())
    cmap = plt.cm.get_cmap(cmap_name, len(makes))
    palette = {make: cmap(i) for i, make in enumerate(makes)}
    return palette


def plot_focus_scatter_with_background(
    ax,
    focus_df,
    background_df,
    make_palette,
    title,
    background_label
):
    """
    focus_df: points to be highlighted by Make
    background_df: the other split, shown in faint gray
    """
    # Background points: very transparent gray
    ax.scatter(
        background_df["actual_pf_ratio"],
        background_df["predicted_pf_ratio"],
        s=38,
        c="gray",
        alpha=0.12,
        edgecolors="none",
        label=background_label
    )

    # Focus points: color by Make
    for make_value, sub_df in focus_df.groupby("make_plot"):
        ax.scatter(
            sub_df["actual_pf_ratio"],
            sub_df["predicted_pf_ratio"],
            s=52,
            color=make_palette[make_value],
            alpha=0.85,
            edgecolors="white",
            linewidths=0.45,
            label=make_value
        )

    combined_actual = np.concatenate([
        focus_df["actual_pf_ratio"].values,
        background_df["actual_pf_ratio"].values
    ])

    combined_pred = np.concatenate([
        focus_df["predicted_pf_ratio"].values,
        background_df["predicted_pf_ratio"].values
    ])

    min_val = min(combined_actual.min(), combined_pred.min())
    max_val = max(combined_actual.max(), combined_pred.max())

    margin = 0.01 * (max_val - min_val + 1e-8)
    min_val -= margin
    max_val += margin

    ax.plot(
        [min_val, max_val],
        [min_val, max_val],
        linestyle="--",
        linewidth=1.5,
        color="black"
    )

    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)

    ax.set_xlabel("Actual PF Ratio")
    ax.set_ylabel("Predicted PF Ratio")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)


def plot_loss_and_scheduler(ax, train_losses, val_losses, head_lrs, backbone_lrs):
    epochs = np.arange(1, len(train_losses) + 1)

    # Left axis: loss
    line1 = ax.plot(
        epochs,
        train_losses,
        marker="o",
        linewidth=2.0,
        label="Train Loss"
    )

    line2 = ax.plot(
        epochs,
        val_losses,
        marker="s",
        linewidth=2.0,
        label="Validation Loss"
    )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss and Scheduler")
    ax.grid(True, alpha=0.25)

    # Right axis: learning rate
    ax2 = ax.twinx()

    line3 = ax2.plot(
        epochs,
        head_lrs,
        linestyle="--",
        marker="o",
        linewidth=1.8,
        label="Head LR"
    )

    line4 = ax2.plot(
        epochs,
        backbone_lrs,
        linestyle="--",
        marker="s",
        linewidth=1.8,
        label="Backbone LR"
    )

    ax2.set_ylabel("Learning Rate")

    lines = line1 + line2 + line3 + line4
    labels = [line.get_label() for line in lines]
    ax.legend(lines, labels, loc="upper right", frameon=True)


def make_prediction_table(base_df, trues, preds, split_name):
    out = base_df.reset_index(drop=True).copy()
    out["actual_pf_ratio"] = trues
    out["predicted_pf_ratio"] = preds
    out["split_used_for_eval"] = split_name
    return out


# =========================================================
# MAIN
# =========================================================
def main():
    df = prepare_dataframe(CSV_PATH)

    if len(df) < 20:
        raise ValueError("Too few valid samples remain after filtering.")

    print("\nSample rows:")
    print(df.head())

    print("\nSplit counts after filtering:")
    print(df["split"].value_counts())

    print("\nMake x split table after filtering:")
    print(pd.crosstab(df["make_plot"], df["split"]))

    train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
    val_df = df[df["split"] == "val"].copy().reset_index(drop=True)
    test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError("One of train/val/test is empty after filtering.")

    print(f"\nTrain samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    print(f"Test samples:  {len(test_df)}")

    train_dataset = DualELDataset(
        train_df,
        image_size=IMAGE_SIZE,
        augment=True,
        preload=PRELOAD_TO_RAM
    )

    val_dataset = DualELDataset(
        val_df,
        image_size=IMAGE_SIZE,
        augment=False,
        preload=PRELOAD_TO_RAM
    )

    test_dataset = DualELDataset(
        test_df,
        image_size=IMAGE_SIZE,
        augment=False,
        preload=PRELOAD_TO_RAM
    )

    pin_memory = True if DEVICE.type == "cuda" else False

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory
    )

    model = build_efficientnet_b1_2ch(pretrained=True).to(DEVICE)

    print("Model is on:", next(model.parameters()).device)

    backbone_params = list(model.features.parameters())
    head_params = list(model.classifier.parameters())

    print(f"\nBackbone parameters: {sum(p.numel() for p in backbone_params):,}")
    print(f"Head parameters:     {sum(p.numel() for p in head_params):,}")
    print(f"Total parameters:    {sum(p.numel() for p in model.parameters()):,}")
    print(f"Backbone LR: {BACKBONE_LR:.1e} | Head LR: {LEARNING_RATE:.1e}")

    criterion = nn.SmoothL1Loss()

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": BACKBONE_LR},
            {"params": head_params, "lr": LEARNING_RATE},
        ],
        weight_decay=1e-2
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_rmse = float("inf")

    train_losses = []
    val_losses = []

    head_lrs = []
    backbone_lrs = []

    print(f"\n{'=' * 78}")
    print("Starting training: EfficientNet-B1, dual low/high EL, fixed split CSV")
    print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE} | Batch size: {BATCH_SIZE}")
    print(f"{'=' * 78}\n")

    total_t0 = time.time()

    for epoch in range(NUM_EPOCHS):
        epoch_t0 = time.time()

        current_backbone_lr = optimizer.param_groups[0]["lr"]
        current_head_lr = optimizer.param_groups[1]["lr"]

        backbone_lrs.append(current_backbone_lr)
        head_lrs.append(current_head_lr)

        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE
        )

        train_eval_loss, train_mae, train_rmse, train_r2, _, _ = evaluate(
            model,
            train_loader,
            criterion,
            DEVICE
        )

        val_loss, val_mae, val_rmse, val_r2, _, _ = evaluate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        scheduler.step(val_loss)

        train_losses.append(train_eval_loss)
        val_losses.append(val_loss)

        epoch_time = time.time() - epoch_t0

        print(
            f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] | "
            f"{epoch_time:.1f}s | "
            f"BB_LR: {current_backbone_lr:.2e} | "
            f"Head_LR: {current_head_lr:.2e} | "
            f"Train Loss: {train_eval_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Train MAE: {train_mae:.6f} | "
            f"Val MAE: {val_mae:.6f} | "
            f"Val RMSE: {val_rmse:.6f} | "
            f"Val R2: {val_r2:.6f}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_model_wts = copy.deepcopy(model.state_dict())

    total_time = time.time() - total_t0

    print(f"\nTraining complete in {total_time:.1f}s ({total_time / 60:.1f} min)")

    # =====================================================
    # FINAL EVALUATION
    # =====================================================
    model.load_state_dict(best_model_wts)

    val_loss, val_mae, val_rmse, val_r2, val_preds, val_trues = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    test_loss, test_mae, test_rmse, test_r2, test_preds, test_trues = evaluate(
        model,
        test_loader,
        criterion,
        DEVICE
    )

    print("\nBest Model Performance selected by Validation RMSE")

    print("\nValidation Set")
    print(f"MAE  : {val_mae:.6f}")
    print(f"RMSE : {val_rmse:.6f}")
    print(f"R2   : {val_r2:.6f}")

    print("\nTest Set")
    print(f"MAE  : {test_mae:.6f}")
    print(f"RMSE : {test_rmse:.6f}")
    print(f"R2   : {test_r2:.6f}")

    # =====================================================
    # PREPARE RESULT TABLES FOR PLOTTING
    # =====================================================
    val_results_df = make_prediction_table(val_df, val_trues, val_preds, "val")
    test_results_df = make_prediction_table(test_df, test_trues, test_preds, "test")

    make_palette = build_make_palette(df["make_plot"], cmap_name="tab20")

    # =====================================================
    # PLOTTING
    # =====================================================
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))

    # 1) Validation scatter, test as faint gray background
    plot_focus_scatter_with_background(
        axes[0],
        focus_df=val_results_df,
        background_df=test_results_df,
        make_palette=make_palette,
        title="Validation: Predicted vs Actual PF Ratio",
        background_label="Test (background)"
    )

    # 2) Test scatter, validation as faint gray background
    plot_focus_scatter_with_background(
        axes[1],
        focus_df=test_results_df,
        background_df=val_results_df,
        make_palette=make_palette,
        title="Test: Predicted vs Actual PF Ratio",
        background_label="Validation (background)"
    )

    # 3) Training loss + scheduler
    plot_loss_and_scheduler(
        axes[2],
        train_losses=train_losses,
        val_losses=val_losses,
        head_lrs=head_lrs,
        backbone_lrs=backbone_lrs
    )

    # Global legend for Make
    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            color="gray",
            linestyle="None",
            markersize=7,
            alpha=0.35,
            label="Background split"
        )
    ]

    for make_value in sorted(make_palette.keys()):
        legend_handles.append(
            Line2D(
                [0], [0],
                marker="o",
                color=make_palette[make_value],
                linestyle="None",
                markersize=7,
                label=make_value
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
        title="Make"
    )

    plt.tight_layout(rect=[0, 0, 0.84, 1])

    result_fig_path = "results_effnet_b1_dual_el_fixedsplit_makecolours.png"
    plt.savefig(result_fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    # =====================================================
    # SAVE PREDICTIONS
    # =====================================================
    val_csv_path = "validation_predictions_effnet_b1_dual_el_fixedsplit.csv"
    test_csv_path = "test_predictions_effnet_b1_dual_el_fixedsplit.csv"

    val_results_df.to_csv(val_csv_path, index=False)
    test_results_df.to_csv(test_csv_path, index=False)

    save_path = "effnet_b1_dual_el_fixedsplit_best_val_model.pth"
    torch.save(model.state_dict(), save_path)

    print("\nSaved:")
    print(f"- {result_fig_path}")
    print(f"- {val_csv_path}")
    print(f"- {test_csv_path}")
    print(f"- {save_path}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nTraining stopped manually by user.")
    finally:
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        print("Cleanup done.")